# Task finale — Valutazione del modello su real_settings


## 1. Caricamento dei 3 file

In [6]:
import pandas as pd

actions  = pd.read_csv("../data/actions - real settings 4.tsv", sep="\t")
features = pd.read_csv("../data/features - real settings 4.tsv", sep="\t")
labels = pd.read_csv("../data/labels - real settings 4.tsv", sep=r"\s+")

print("actions: ", actions.shape)
print("features:", features.shape)
print("labels:  ", labels.shape)

print(labels.head())
print(labels.tail())

actions:  (100, 4)
features: (100, 5)
labels:   (100, 2)
   ACTIONID  LABEL
0       101      0
1       102      0
2       103      0
3       104      1
4       105      0
    ACTIONID  LABEL
95       196      0
96       197      0
97       198      0
98       199      0
99       200      1


## 2. Merge su ACTIONID

In [7]:
df = actions.merge(features, on="ACTIONID").merge(labels, on="ACTIONID")

print("righe dopo il merge:", len(df))
df.head()

righe dopo il merge: 100


,ACTIONID,USERID,TARGETID,TIMESTAMP,FEATURE0,FEATURE1,FEATURE2,FEATURE3,LABEL
0,101,14,8,39328.0,-0.319991,-0.435701,0.106784,-0.067309,0
1,102,19,13,39329.0,-0.319991,2.108722,-0.394237,-0.067309,0
2,103,14,8,39364.0,-0.319991,-0.435701,0.106784,-0.067309,0
3,104,14,1,39370.0,-0.319991,-0.435701,0.106784,-0.067309,1
4,105,10,4,39382.0,-0.319991,-0.435701,0.106784,-0.067309,0


## 3. Pulizia e controllo dataset


In [ ]:
# valori mancanti e range
print(df.isna().sum())

# righe duplicate 
print("duplicati esatti:", df.duplicated().sum())
df_clean = df.drop_duplicates()

# contraddizioni
stesso_id = df_clean["ACTIONID"].duplicated(keep=False)
riga_diversa = ~df_clean.duplicated(keep=False)
anomalie = df_clean[stesso_id & riga_diversa].sort_values("ACTIONID")
print("righe contraddittorie:", len(anomalie))
print(anomalie[["ACTIONID", "LABEL"]])
df_clean = df_clean.drop(anomalie.index)

print("righe finali:", len(df_clean))


ACTIONID     0
USERID       0
TARGETID     0
TIMESTAMP    0
FEATURE0     0
FEATURE1     0
FEATURE2     0
FEATURE3     0
LABEL        0
dtype: int64
duplicati esatti: 6
righe contraddittorie: 4
    ACTIONID  LABEL
46       148      0
47       148      1
88       190      1
89       190      0
righe finali: 90


## 4. Implementazione DecisionTree

In [9]:
from sklearn.tree import DecisionTreeClassifier

train = pd.read_csv("../data/training_clean.csv")
FEATS = ["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3"]
X_train = train[FEATS]
y_train = train["LABEL"]

# il classificatore finale scelto 
modello = DecisionTreeClassifier(class_weight="balanced", max_depth=2,
                                 min_samples_leaf=1, random_state=42)
modello.fit(X_train, y_train)

print("modello addestrato su", len(train), "righe")

modello addestrato su 49799 righe


## 5. Predizione 

In [10]:
# applichiamo il modello 
df_clean["PREDETTA"] = modello.predict(df_clean[FEATS])

print(df_clean["PREDETTA"].value_counts())


PREDETTA
0    65
1    25
Name: count, dtype: int64


## 6. Valutazione

In [11]:
TP = ((df_clean["LABEL"] == 1) & (df_clean["PREDETTA"] == 1)).sum()
TN = ((df_clean["LABEL"] == 0) & (df_clean["PREDETTA"] == 0)).sum()
FP = ((df_clean["LABEL"] == 0) & (df_clean["PREDETTA"] == 1)).sum()
FN = ((df_clean["LABEL"] == 1) & (df_clean["PREDETTA"] == 0)).sum()

print(pd.DataFrame([[TN, FP], [FN, TP]],
                   index=["Reale 0", "Reale 1"],
                   columns=["Predetta 0", "Predetta 1"]))
print()

accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

         Predetta 0  Predetta 1
Reale 0          59          20
Reale 1           6           5

Accuracy : 0.7111
Precision: 0.2000
Recall   : 0.4545
F1-score : 0.2778


## 7. Commento dei risultati
* **Cosa fa bene:** Riconosce abbastanza la classe 0.
* **Dove pecca:** Sulla classe di reale interesse 1, indovina solo **5 volte su 11** (bassa Recall) e ben l'**80% delle sue predizioni positive sono falsi allarmi** (bassa Precision).

